<a href="https://colab.research.google.com/github/redwan-nabil/C-programming/blob/main/colab_t4_stress_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1 - hardware info (Runtime > Change runtime type > T4 GPU first)
import subprocess, psutil, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
vm = psutil.virtual_memory()
print(f"CPU cores: {psutil.cpu_count()} | RAM total: {vm.total/2**30:.1f} GiB | free: {vm.available/2**30:.1f} GiB")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())


Sun Nov 15 07:06:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# CELL 2 - config + helpers
import os, gc, time, threading, subprocess, multiprocessing as mp
import numpy as np, psutil, torch

# ---------------- CONFIG ----------------
DURATION   = 180            # seconds of full load (keep it a few minutes)
MATRIX_N   = 8192           # matmul size
GPU_DTYPE  = torch.float16  # float16 = tensor cores (max TFLOPS) | torch.float32 = plain CUDA cores
VRAM_FILL  = 0.92           # fraction of free VRAM to occupy
RAM_FILL   = 0.90           # fraction of total CPU RAM to fill (>0.95 usually kills the Colab runtime)
USE_CPU    = True           # burn all CPU cores too
# ----------------------------------------

REASONS = {0x2: "app-clocks", 0x4: "SW power cap", 0x8: "HW slowdown",
           0x20: "SW thermal", 0x40: "HW thermal", 0x80: "HW power brake"}
_FIELDS = ["temperature.gpu", "clocks.sm", "clocks.max.sm", "power.draw", "power.limit",
           "utilization.gpu", "memory.used"]
_reason_field = None

def _num(x):
    try: return float(x)
    except: return 0.0

def gpu_stats():
    global _reason_field
    names = [_reason_field] if _reason_field else ["clocks_throttle_reasons.active", "clocks_event_reasons.active"]
    for rf in names:
        r = subprocess.run(["nvidia-smi", "--query-gpu=" + ",".join(_FIELDS + [rf]),
                            "--format=csv,noheader,nounits"], capture_output=True, text=True)
        if r.returncode == 0:
            _reason_field = rf
            v = [x.strip() for x in r.stdout.strip().split(",")]
            try: mask = int(v[7], 16)
            except: mask = 0
            return dict(temp=_num(v[0]), sm=_num(v[1]), smmax=_num(v[2]), pw=_num(v[3]),
                        pl=_num(v[4]), util=_num(v[5]), vram=_num(v[6]),
                        why=[n for b, n in REASONS.items() if mask & b])
    raise RuntimeError("nvidia-smi query failed")

class Monitor(threading.Thread):
    def __init__(self, stop, every=5):
        super().__init__(daemon=True)
        self.stop_ev, self.every, self.t0 = stop, every, time.time()
        self.peak_temp = self.peak_pw = self.peak_ram = 0
        self.min_clk, self.reasons = 10**9, set()
    def run(self):
        while not self.stop_ev.is_set():
            vm = psutil.virtual_memory()
            ram = (vm.total - vm.available) / 2**30
            self.peak_ram = max(self.peak_ram, ram)
            try:
                s = gpu_stats()
                self.peak_temp = max(self.peak_temp, s["temp"]); self.peak_pw = max(self.peak_pw, s["pw"])
                if s["util"] > 90: self.min_clk = min(self.min_clk, s["sm"])
                self.reasons.update(s["why"])
                print(f"[{time.time()-self.t0:5.0f}s] GPU {s['temp']:.0f}C {s['sm']:.0f}/{s['smmax']:.0f}MHz "
                      f"{s['pw']:.0f}/{s['pl']:.0f}W util {s['util']:.0f}% VRAM {s['vram']/1024:.1f}GiB "
                      f"throttle={s['why'] or 'none'} | CPU {psutil.cpu_percent():.0f}% RAM {ram:.1f}GiB", flush=True)
            except Exception as e:
                print("monitor error:", e)
            self.stop_ev.wait(self.every)

def cpu_burn(deadline):
    a = np.random.rand(1500, 1500)
    while time.time() < deadline:
        a = a @ a
        a /= np.abs(a).max()

def fill_ram(frac):
    total = psutil.virtual_memory().total
    chunks = []
    while True:
        vm = psutil.virtual_memory()
        if (total - vm.available) >= total * frac: break
        chunks.append(np.ones(32 * 2**20, dtype=np.float64))   # 256 MiB, pages touched
    return chunks

def ram_hammer(chunks, stop, out):
    n, t = 0, time.time()
    while not stop.is_set():
        for c in chunks:
            np.multiply(c, 1.0000001, out=c)        # read + write
            n += c.nbytes * 2
            if stop.is_set(): break
    out["gbps"] = n / (time.time() - t) / 1e9
print("helpers ready")


helpers ready


In [3]:
# CELL 3 - FULL LOAD: GPU (tensor cores + VRAM bandwidth + VRAM full) + all CPU cores + RAM full
stop = threading.Event()
procs = []
if USE_CPU:                                   # fork CPU workers before CUDA is initialised
    ctx = mp.get_context("fork")
    dl = time.time() + DURATION + 15
    procs = [ctx.Process(target=cpu_burn, args=(dl,), daemon=True) for _ in range(os.cpu_count())]
    [p.start() for p in procs]

print(f"Filling CPU RAM to {RAM_FILL*100:.0f}% ...")
chunks = fill_ram(RAM_FILL)
ram_stats = {}
ram_t = threading.Thread(target=ram_hammer, args=(chunks, stop, ram_stats), daemon=True); ram_t.start()

dev = "cuda"
free, total = torch.cuda.mem_get_info()
N = MATRIX_N
a = torch.randn(N, N, device=dev, dtype=GPU_DTYPE) * 0.01
b = torch.randn(N, N, device=dev, dtype=GPU_DTYPE) * 0.01
c = torch.empty(N, N, device=dev, dtype=GPU_DTYPE)
free, _ = torch.cuda.mem_get_info()
bufsz = int(free * 0.25)
buf1 = torch.empty(bufsz, dtype=torch.uint8, device=dev); buf2 = torch.empty_like(buf1)
ballast = []
while torch.cuda.mem_get_info()[0] > total * (1 - VRAM_FILL) + 512 * 2**20:
    ballast.append(torch.empty(512 * 2**20, dtype=torch.uint8, device=dev))
print(f"VRAM allocated: {(total - torch.cuda.mem_get_info()[0])/2**30:.1f} / {total/2**30:.1f} GiB")

mon = Monitor(stop); mon.start()
hist, it, cp = [], 0, 0
t0 = time.time()
try:
    while time.time() - t0 < DURATION:
        ts = time.time()
        for _ in range(20): torch.matmul(a, b, out=c)
        buf2.copy_(buf1); buf1.copy_(buf2)
        torch.cuda.synchronize()
        dt = time.time() - ts
        it += 20; cp += 2
        hist.append((ts - t0, 20 * 2 * N**3 / dt / 1e12))
finally:
    stop.set()
    elapsed = time.time() - t0
    ram_t.join(timeout=10)
    for p in procs: p.terminate()

q = max(1, len(hist) // 4)
first = np.mean([h[1] for h in hist[:q]]); last = np.mean([h[1] for h in hist[-q:]])
print("\n=========== RESULTS ===========")
print(f"GPU {str(GPU_DTYPE).split('.')[-1]} matmul : avg {it*2*N**3/elapsed/1e12:.1f} TFLOPS  (first quarter {first:.1f} -> last quarter {last:.1f}, drop {100*(1-last/first):.1f}%)")
print(f"VRAM copy bandwidth : {cp*bufsz*2/elapsed/1e9:.0f} GB/s (T4 theoretical ~320)")
print(f"CPU RAM hammer      : {ram_stats.get('gbps', 0):.1f} GB/s | peak RAM used {mon.peak_ram:.1f} GiB")
print(f"Peak GPU temp {mon.peak_temp:.0f}C | peak power {mon.peak_pw:.0f}W | lowest SM clock under load {mon.min_clk:.0f} MHz")
print("Throttle reasons seen:", sorted(mon.reasons) or "none")

# free everything
del a, b, c, buf1, buf2, ballast, chunks
gc.collect(); torch.cuda.empty_cache()


Filling CPU RAM to 90% ...
VRAM allocated: 13.0 / 14.6 GiB
[    0s] GPU 46C 585/1590MHz 27/70W util 4% VRAM 13.1GiB throttle=none | CPU 78% RAM 11.6GiB
[    5s] GPU 53C 645/1590MHz 68/70W util 100% VRAM 13.1GiB throttle=['SW power cap'] | CPU 100% RAM 11.6GiB
[   11s] GPU 56C 660/1590MHz 68/70W util 100% VRAM 13.1GiB throttle=['SW power cap'] | CPU 100% RAM 11.6GiB
[   16s] GPU 60C 750/1590MHz 58/70W util 94% VRAM 13.1GiB throttle=['SW power cap'] | CPU 100% RAM 11.6GiB
[   21s] GPU 61C 630/1590MHz 67/70W util 100% VRAM 13.1GiB throttle=['SW power cap'] | CPU 100% RAM 11.6GiB
[   26s] GPU 63C 615/1590MHz 68/70W util 100% VRAM 13.1GiB throttle=['SW power cap'] | CPU 100% RAM 11.6GiB
[   31s] GPU 65C 630/1590MHz 68/70W util 100% VRAM 13.1GiB throttle=['SW power cap'] | CPU 100% RAM 11.7GiB
[   37s] GPU 66C 615/1590MHz 67/70W util 100% VRAM 13.1GiB throttle=['SW power cap'] | CPU 100% RAM 11.6GiB
[   42s] GPU 68C 615/1590MHz 68/70W util 100% VRAM 13.1GiB throttle=['SW power cap'] | CPU 10